In [ ]:
!pip install optuna lightgbm clickhouse-connect scikit-learn plotly matplotlib seaborn pandas numpy

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import uuid
import random
from collections import defaultdict
import datetime as dt

import pandas as pd
import numpy as np
import clickhouse_connect

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import lightgbm as lgb
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    fbeta_score, confusion_matrix, classification_report
)
from sklearn.utils import resample

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

torch.manual_seed(42)
np.random.seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
faulty_ids = []

In [ ]:
client = clickhouse_connect.get_client(
    host='localhost', port=8123, username='default', password='', database='default'
)

## EDA

In [ ]:
def plot_telemetry(client, channel_id: str):
    query = """
        SELECT date_time, value
        FROM telemetry
        WHERE channel_id = {ch_id:UUID}
          AND date_time >= '2025-12-15 00:00:00'
          AND date_time <= '2026-05-10 23:59:59'
        ORDER BY date_time
    """
    df = client.query_df(query, parameters={'ch_id': channel_id})
    if df.empty:
        return
    
    fig = px.line(df, x='date_time', y='value', title=f'Телеметрия: {channel_id}')
    fig.update_xaxes(
        rangeslider_visible=True,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1м", step="month", stepmode="backward"),
                dict(count=6, label="6м", step="month", stepmode="backward"),
                dict(step="all", label="Все")
            ])
        )
    )
    fig.update_traces(line_color='royalblue')
    fig.show()

In [ ]:
plot_telemetry(client, '')

In [ ]:
def plot_advanced_eda(client, channel_id: str):
    query = """
        SELECT 
            toStartOfHour(date_time) AS hour_start,
            avgIf(value, value < 327.0) AS mean_val,
            stddevSampIf(value, value < 327.0) AS std_val,
            maxIf(value, value < 327.0) - minIf(value, value < 327.0) AS peak_to_peak,
            quantileIf(0.95)(value, value < 327.0) AS p95_val,
            quantileIf(0.99)(value, value < 327.0) AS p99_val,
            countIf(value >= 327.66 AND value <= 327.67) AS error_codes_count
        FROM telemetry
        WHERE channel_id = {ch_id:UUID}
          AND date_time >= '2025-12-15 00:00:00'
          AND date_time <= '2026-05-10 23:59:59'
        GROUP BY hour_start
        ORDER BY hour_start
    """
    df = client.query_df(query, parameters={'ch_id': channel_id})
    if df.empty: return
    df['std_val'] = df['std_val'].fillna(0)

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
        subplot_titles=("1. Тренд и p99", "2. Peak-to-Peak", "3. Стандартное отклонение", "4. Ошибки КЗ")
    )
    fig.add_trace(go.Scatter(x=df['hour_start'], y=df['mean_val'], name='Mean', line=dict(color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['hour_start'], y=df['p99_val'], name='99th Percentile', line=dict(color='rgba(255, 0, 0, 0.4)', dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['hour_start'], y=df['peak_to_peak'], name='Peak-to-Peak', line=dict(color='orange')), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['hour_start'], y=df['std_val'], name='Std Dev', line=dict(color='purple')), row=3, col=1)
    fig.add_trace(go.Bar(x=df['hour_start'], y=df['error_codes_count'], name='Error Codes (327.6x)', marker_color='red'), row=4, col=1)

    fig.update_layout(title=f"EDA метрик деградации: {channel_id}", height=900, hovermode="x unified", showlegend=True)
    fig.show()

## Feature engineering

In [ ]:
def fix_uuids(val):
    if isinstance(val, bytes):
        try:
            return str(uuid.UUID(bytes=val))
        except Exception:
            return str(val)
    return str(val)

def prepare_ml_dataset_sampled_dynamic(client, faulty_ids: list, n_healthy: int = 500):
    all_ids_df = client.query_df("SELECT DISTINCT channel_id FROM telemetry")
    all_ids_df['channel_id'] = all_ids_df['channel_id'].apply(fix_uuids)
    all_ids = all_ids_df['channel_id'].unique().tolist()
    
    faulty_ids_str = list(set([fix_uuids(fid) for fid in faulty_ids]))
    healthy_ids = [cid for cid in all_ids if cid not in faulty_ids_str]
    sampled_healthy = random.sample(healthy_ids, min(n_healthy, len(healthy_ids)))
    selected_ids = faulty_ids_str + sampled_healthy

    query = """
        SELECT 
            channel_id, toStartOfHour(date_time) AS timestamp,
            avgIf(value, value < 327.0) AS mean_1h,
            stddevSampIf(value, value < 327.0) AS std_1h,
            minIf(value, value < 327.0) AS min_1h,
            maxIf(value, value < 327.0) AS max_1h,
            quantileIf(0.95)(value, value < 327.0) AS p95_1h,
            quantileIf(0.99)(value, value < 327.0) AS p99_1h,
            countIf(value >= 327.66 AND value <= 327.67) AS errors_1h
        FROM telemetry
        WHERE channel_id IN {selected_ids:Array(UUID)}
          AND date_time >= '2025-12-15 00:00:00' AND date_time <= '2026-05-13 23:59:59'
        GROUP BY channel_id, timestamp
        ORDER BY channel_id, timestamp
    """
    df = client.query_df(query, parameters={'selected_ids': selected_ids})
    df['channel_id'] = df['channel_id'].apply(fix_uuids)
    df['std_1h'] = df['std_1h'].fillna(0)
    
    df = df.set_index('timestamp')
    
    def process_sensor(group):
        full_idx = pd.date_range(start=group.index.min(), end=group.index.max(), freq='h')
        group = group.reindex(full_idx)
        group['is_missing_data'] = group['mean_1h'].isna().astype(int)
        group = group.ffill(limit=24) 
        group['channel_id'] = group['channel_id'].ffill()
        group['errors_1h'] = group['errors_1h'].fillna(0)
        return group

    df = df.groupby('channel_id', as_index=False, group_keys=False).apply(process_sensor)
    df = df.reset_index(names=['timestamp']) if df.index.name != 'timestamp' else df.reset_index()

    df = df.sort_values(['channel_id', 'timestamp'])
    gb = df.groupby('channel_id')
    
    df['std_12h_mean'] = gb['std_1h'].transform(lambda x: x.rolling(12, min_periods=1).mean())
    df['std_24h_mean'] = gb['std_1h'].transform(lambda x: x.rolling(24, min_periods=1).mean())
    df['std_3d_mean']  = gb['std_1h'].transform(lambda x: x.rolling(24*3, min_periods=1).mean())
    df['std_7d_mean']  = gb['std_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).mean())
    
    df['std_24h_max'] = gb['std_1h'].transform(lambda x: x.rolling(24, min_periods=1).max())
    df['std_3d_max']  = gb['std_1h'].transform(lambda x: x.rolling(24*3, min_periods=1).max())
    df['std_7d_max']  = gb['std_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).max())
    df['std_14d_max'] = gb['std_1h'].transform(lambda x: x.rolling(24*14, min_periods=1).max())

    df['mean_24h_mean'] = gb['mean_1h'].transform(lambda x: x.rolling(24, min_periods=1).mean())
    df['mean_7d_mean']  = gb['mean_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).mean())
    df['trend_diff_7d'] = df['mean_1h'] - df['mean_7d_mean']
    df['errors_24h_sum'] = gb['errors_1h'].transform(lambda x: x.rolling(24, min_periods=1).sum())
    df['errors_7d_sum']  = gb['errors_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).sum())
    df['peak_to_peak_1h'] = df['max_1h'] - df['min_1h']

    df['is_negative_temp'] = (df['min_1h'] < 0).astype(int)
    df['negative_hours_last_7d'] = gb['is_negative_temp'].transform(lambda x: x.rolling(24*7, min_periods=1).sum())
    
    def calc_advanced_baseline(group):
        baseline_period = group.head(24 * 30) 
        med_std = baseline_period['std_1h'].median()
        mad_std = np.median(np.abs(baseline_period['std_1h'] - med_std))
        med_temp = baseline_period['mean_1h'].median()
        if mad_std == 0: mad_std = 0.001 
        if med_std == 0: med_std = 0.001
        return pd.Series({'med_std': med_std, 'mad_std': mad_std, 'med_temp': med_temp})

    baselines = df.groupby('channel_id').apply(calc_advanced_baseline).to_dict(orient='index')
    
    def apply_advanced_features(row):
        base = baselines.get(row['channel_id'], {'med_std': 0.001, 'mad_std': 0.001, 'med_temp': 0})
        z_score = 0.6745 * (row['std_1h'] - base['med_std']) / base['mad_std']
        std_shift_ratio = row['std_3d_mean'] / base['med_std']
        temp_drop_amplitude = max(0, base['med_temp'] - row['min_1h']) 
        return pd.Series({'std_robust_z_score': z_score, 'std_shift_ratio': std_shift_ratio, 'temp_drop_amplitude': temp_drop_amplitude})
                          
    df = pd.concat([df, df.apply(apply_advanced_features, axis=1)], axis=1)
    df['z_score_24h_mean'] = df.groupby('channel_id')['std_robust_z_score'].transform(lambda x: x.rolling(24, min_periods=1).mean())

    failures = df[df['errors_24h_sum'] >= 5].groupby('channel_id')['timestamp'].min().reset_index()
    failures.columns = ['channel_id', 'failure_time']
    df = pd.merge(df, failures, on='channel_id', how='left')
    df['target'] = 0 
    
    for sid in failures['channel_id'].unique():
        sensor_mask = df['channel_id'] == sid
        f_time = df.loc[sensor_mask, 'failure_time'].iloc[0]
        max_horizon_time = f_time - pd.Timedelta(days=60)
        recent_data = df[sensor_mask & (df['timestamp'] >= max_horizon_time) & (df['timestamp'] < f_time)]
        breakdowns = recent_data[recent_data['std_robust_z_score'] > 5.0]
        
        start_time = breakdowns['timestamp'].min() if not breakdowns.empty else f_time - pd.Timedelta(days=14)
        mask_pre_failure = sensor_mask & (df['timestamp'] >= start_time) & (df['timestamp'] <= (f_time - pd.Timedelta(hours=1)))
        df.loc[mask_pre_failure, 'target'] = 1

    df = df[~(df['failure_time'].notna() & (df['timestamp'] >= df['failure_time']))]
    df = df.drop(columns=['failure_time']).dropna()
    return df

In [ ]:
df = prepare_ml_dataset_sampled_dynamic(client, faulty_ids)

In [ ]:
def prepare_unseen_healthy_dataset(client, faulty_ids, training_df, n_healthy=500):
    training_ids = training_df['channel_id'].unique().tolist()
    faulty_ids_str = list(set([fix_uuids(fid) for fid in faulty_ids]))
    used_ids_str = list(set([fix_uuids(fid) for fid in training_ids]))
    
    all_ids_df = client.query_df("SELECT DISTINCT channel_id FROM telemetry")
    all_ids_df['channel_id'] = all_ids_df['channel_id'].apply(fix_uuids)
    all_ids = all_ids_df['channel_id'].unique().tolist()
    
    unseen_healthy_ids = [cid for cid in all_ids if cid not in faulty_ids_str and cid not in used_ids_str]
    sampled_unseen = random.sample(unseen_healthy_ids, min(n_healthy, len(unseen_healthy_ids)))

    query = """
        SELECT 
            channel_id, toStartOfHour(date_time) AS timestamp,
            avgIf(value, value < 327.0) AS mean_1h,
            stddevSampIf(value, value < 327.0) AS std_1h,
            minIf(value, value < 327.0) AS min_1h,
            maxIf(value, value < 327.0) AS max_1h,
            quantileIf(0.95)(value, value < 327.0) AS p95_1h,
            quantileIf(0.99)(value, value < 327.0) AS p99_1h,
            countIf(value >= 327.66 AND value <= 327.67) AS errors_1h
        FROM telemetry
        WHERE channel_id IN {selected_ids:Array(UUID)}
          AND date_time >= '2025-12-15 00:00:00' AND date_time <= '2026-05-13 23:59:59'
        GROUP BY channel_id, timestamp
        ORDER BY channel_id, timestamp
    """
    
    unseen_df = client.query_df(query, parameters={'selected_ids': sampled_unseen})
    unseen_df['channel_id'] = unseen_df['channel_id'].apply(fix_uuids)
    unseen_df['std_1h'] = unseen_df['std_1h'].fillna(0)
    
    unseen_df = unseen_df.set_index('timestamp')
    
    def process_sensor(group):
        full_idx = pd.date_range(start=group.index.min(), end=group.index.max(), freq='h')
        group = group.reindex(full_idx)
        group['is_missing_data'] = group['mean_1h'].isna().astype(int)
        group = group.ffill(limit=24) 
        group['channel_id'] = group['channel_id'].ffill()
        group['errors_1h'] = group['errors_1h'].fillna(0)
        return group

    unseen_df = unseen_df.groupby('channel_id', as_index=False, group_keys=False).apply(process_sensor)
    unseen_df = unseen_df.reset_index(names=['timestamp']) if unseen_df.index.name != 'timestamp' else unseen_df.reset_index()

    unseen_df = unseen_df.sort_values(['channel_id', 'timestamp'])
    gb = unseen_df.groupby('channel_id')
    
    unseen_df['std_12h_mean'] = gb['std_1h'].transform(lambda x: x.rolling(12, min_periods=1).mean())
    unseen_df['std_24h_mean'] = gb['std_1h'].transform(lambda x: x.rolling(24, min_periods=1).mean())
    unseen_df['std_3d_mean']  = gb['std_1h'].transform(lambda x: x.rolling(24*3, min_periods=1).mean())
    unseen_df['std_7d_mean']  = gb['std_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).mean())
    unseen_df['std_24h_max'] = gb['std_1h'].transform(lambda x: x.rolling(24, min_periods=1).max())
    unseen_df['std_3d_max']  = gb['std_1h'].transform(lambda x: x.rolling(24*3, min_periods=1).max())
    unseen_df['std_7d_max']  = gb['std_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).max())
    unseen_df['std_14d_max'] = gb['std_1h'].transform(lambda x: x.rolling(24*14, min_periods=1).max())
    unseen_df['mean_24h_mean'] = gb['mean_1h'].transform(lambda x: x.rolling(24, min_periods=1).mean())
    unseen_df['mean_7d_mean']  = gb['mean_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).mean())
    unseen_df['trend_diff_7d'] = unseen_df['mean_1h'] - unseen_df['mean_7d_mean']
    unseen_df['errors_24h_sum'] = gb['errors_1h'].transform(lambda x: x.rolling(24, min_periods=1).sum())
    unseen_df['errors_7d_sum']  = gb['errors_1h'].transform(lambda x: x.rolling(24*7, min_periods=1).sum())
    unseen_df['peak_to_peak_1h'] = unseen_df['max_1h'] - unseen_df['min_1h']
    unseen_df['is_negative_temp'] = (unseen_df['min_1h'] < 0).astype(int)
    unseen_df['negative_hours_last_7d'] = gb['is_negative_temp'].transform(lambda x: x.rolling(24*7, min_periods=1).sum())
    
    def calc_advanced_baseline(group):
        baseline_period = group.head(24 * 30) 
        med_std = baseline_period['std_1h'].median()
        mad_std = np.median(np.abs(baseline_period['std_1h'] - med_std))
        med_temp = baseline_period['mean_1h'].median()
        if mad_std == 0: mad_std = 0.001 
        if med_std == 0: med_std = 0.001
        return pd.Series({'med_std': med_std, 'mad_std': mad_std, 'med_temp': med_temp})

    baselines = unseen_df.groupby('channel_id').apply(calc_advanced_baseline).to_dict(orient='index')
    
    def apply_advanced_features(row):
        base = baselines.get(row['channel_id'], {'med_std': 0.001, 'mad_std': 0.001, 'med_temp': 0})
        z_score = 0.6745 * (row['std_1h'] - base['med_std']) / base['mad_std']
        std_shift_ratio = row['std_3d_mean'] / base['med_std']
        temp_drop_amplitude = max(0, base['med_temp'] - row['min_1h'])
        return pd.Series({'std_robust_z_score': z_score, 'std_shift_ratio': std_shift_ratio, 'temp_drop_amplitude': temp_drop_amplitude})
                          
    unseen_df = pd.concat([unseen_df, unseen_df.apply(apply_advanced_features, axis=1)], axis=1)
    unseen_df['z_score_24h_mean'] = unseen_df.groupby('channel_id')['std_robust_z_score'].transform(lambda x: x.rolling(24, min_periods=1).mean())

    unseen_df['target'] = 0
    unseen_df = unseen_df.dropna()
    return unseen_df

In [ ]:
unseen_df = prepare_unseen_healthy_dataset(client, faulty_ids, df)

## Model Training & Validation

In [ ]:
df = df.sort_values(['channel_id', 'timestamp']).reset_index(drop=True)

drop_cols = ['channel_id', 'timestamp', 'target']
base_features = [c for c in df.columns if c not in drop_cols]

X_base = df[base_features]
y = df['target']
groups = df['channel_id']

class_0_count = (y == 0).sum()
class_1_count = (y == 1).sum()
scale_weight = class_0_count / class_1_count

In [ ]:
selected_columns = [col for col in [
    'mean_1h', 'std_1h', 'std_24h_mean', 'std_7d_max', 'errors_24h_sum',
    'std_robust_z_score', 'std_shift_ratio', 'temp_drop_amplitude'
] if col in df.columns] + ['target']

corr_matrix = df[selected_columns].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Корреляционная матрица статистических признаков")
plt.tight_layout()
plt.show()

In [ ]:
def verify_dataset(dataframe):
    broken_sensors = dataframe[dataframe['target'] == 1]['channel_id'].unique().tolist()
    working_sensors = [c for c in dataframe['channel_id'].unique() if c not in broken_sensors]

    sample_broken = random.sample(broken_sensors, min(3, len(broken_sensors)))
    sample_working = random.sample(working_sensors, min(3, len(working_sensors)))

    def plot_sensor(sensor_id, status_title):
        sensor_df = dataframe[dataframe['channel_id'] == sensor_id].sort_values('timestamp')
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        if 'temp_drop_amplitude' in sensor_df.columns:
            fig.add_trace(go.Scatter(x=sensor_df['timestamp'], y=sensor_df['temp_drop_amplitude'], name="temp_drop_amplitude", line=dict(color='purple', width=1)), secondary_y=False)
        if 'std_shift_ratio' in sensor_df.columns:
            fig.add_trace(go.Scatter(x=sensor_df['timestamp'], y=sensor_df['std_shift_ratio'], name="std_shift_ratio", line=dict(color='magenta', width=2, dash='dot')), secondary_y=False)
        if 'std_7d_max' in sensor_df.columns:
            fig.add_trace(go.Scatter(x=sensor_df['timestamp'], y=sensor_df['std_7d_max'], name="std_7d_max", line=dict(color='orange', width=1)), secondary_y=False)

        base_col = 'mean_1h' if 'mean_1h' in sensor_df.columns else sensor_df.columns[2]
        fig.add_trace(go.Scatter(x=sensor_df['timestamp'], y=sensor_df[base_col], name=f"{base_col}", line=dict(color='blue', width=2)), secondary_y=True)

        if 1 in sensor_df['target'].values:
            max_val = sensor_df[base_col].max()
            fig.add_trace(go.Scatter(x=sensor_df['timestamp'], y=sensor_df['target'] * max_val, name="Target Area", line=dict(color='rgba(255,0,0,0)'), fill='tozeroy', fillcolor='rgba(255, 0, 0, 0.2)'), secondary_y=True)

        fig.update_layout(title_text=f"{status_title} | ID: {sensor_id}", height=450, hovermode="x unified", margin=dict(t=50, b=20))
        fig.show()

    for sid in sample_broken: plot_sensor(sid, "🔴 СЛОМАННЫЙ")
    for sid in sample_working: plot_sensor(sid, "🟢 РАБОЧИЙ")

verify_dataset(df)

In [ ]:
RUN_OPTUNA = False

if RUN_OPTUNA:
    def objective(trial):
        param = {
            'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
            'scale_pos_weight': scale_weight,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
            'verbose': -1, 'n_estimators': 300
        }
        gkf = GroupKFold(n_splits=3)
        pr_auc_scores = []

        for train_idx, val_idx in gkf.split(X_base, y, groups):
            X_train, y_train = X_base.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X_base.iloc[val_idx], y.iloc[val_idx]
            model = lgb.LGBMClassifier(**param)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])
            preds = model.predict_proba(X_val)[:, 1]
            pr_auc_scores.append(average_precision_score(y_val, preds))

        return np.mean(pr_auc_scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    for k, v in study.best_params.items(): print(f"'{k}': {v}")

In [ ]:
class AnomalyAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(AnomalyAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 4)
        )
        self.decoder = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, input_dim)
        )
    def forward(self, x): return self.decoder(self.encoder(x))

df_normal = df[df['target'] == 0].copy()
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(df_normal[base_features])
X_all_scaled = scaler.transform(df[base_features].values)

tensor_normal = torch.FloatTensor(X_normal_scaled)
tensor_all = torch.FloatTensor(X_all_scaled)
train_loader = DataLoader(TensorDataset(tensor_normal, tensor_normal), batch_size=512, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ae_model = AnomalyAutoencoder(len(base_features)).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(ae_model.parameters(), lr=0.001)

epochs = 16
ae_model.train()
for epoch in range(epochs):
    train_loss = 0.0
    for batch_x, _ in train_loader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        loss = criterion(ae_model(batch_x), batch_x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_x.size(0)

ae_model.eval()
with torch.no_grad():
    tensor_all_device = tensor_all.to(device)
    predictions = ae_model(tensor_all_device)
    mse_errors = torch.mean((tensor_all_device - predictions) ** 2, dim=1).cpu().numpy()

df['reconstruction_error'] = mse_errors

In [ ]:
hybrid_features = base_features + ['reconstruction_error']
X = df[hybrid_features]

lgb_params = {
    'objective': 'binary', 'boosting_type': 'gbdt', 'scale_pos_weight': scale_weight, 
    'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6, 'feature_fraction': 0.8,
    'min_data_in_leaf': 100, 'n_estimators': 500, 'n_jobs': -1, 'verbose': -1, 'random_state': 42
}

oof_preds = np.zeros(len(X))
gkf = GroupKFold(n_splits=5)
final_models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), 1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='binary_logloss', callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    final_models.append(model)

final_pr_auc = average_precision_score(y, oof_preds)

In [ ]:
precisions_ae, recalls_ae, thresholds_ae = precision_recall_curve(y, mse_errors)
numerator = 5 * precisions_ae[:-1] * recalls_ae[:-1]
denominator = 4 * precisions_ae[:-1] + recalls_ae[:-1] + 1e-10
f2_scores_ae = numerator / denominator

best_idx_ae = np.argmax(f2_scores_ae)
best_threshold_ae = thresholds_ae[best_idx_ae]
y_pred_ae_final = (mse_errors >= best_threshold_ae).astype(int)

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y, oof_preds)
best_f2, best_threshold = 0, 0.5

for t in np.arange(0.1, 0.95, 0.05):
    y_pred_binary = (oof_preds >= t).astype(int)
    f2 = fbeta_score(y, y_pred_binary, beta=2, zero_division=0)
    if f2 > best_f2: best_f2, best_threshold = f2, t

y_pred_final = (oof_preds >= best_threshold).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(confusion_matrix(y, y_pred_final), annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f'Матрица ошибок (Threshold={best_threshold:.2f})')

axes[1].plot(recalls, precisions, marker='.', label='Гибридная модель', color='purple')
axes[1].set_title(f'PR-Кривая (PR-AUC = {final_pr_auc:.3f})')

importances = np.mean([model.feature_importances_ for model in final_models], axis=0)
feat_imp = pd.DataFrame({'Feature': hybrid_features, 'Importance': importances}).sort_values(by='Importance', ascending=False).head(12)
colors = ['red' if feat == 'reconstruction_error' else 'steelblue' for feat in feat_imp['Feature']]
sns.barplot(x='Importance', y='Feature', data=feat_imp, ax=axes[2], palette=colors)
axes[2].set_title('Топ-12 признаков (AE красный)')
plt.tight_layout()
plt.show()

In [ ]:
def calculate_bootstrap_ci(y_true, y_pred_proba, threshold, n_bootstraps=100, ci=95):
    bootstrapped_pr_aucs, bootstrapped_f2_scores = [], []
    y_true_np, y_pred_proba_np = np.array(y_true), np.array(y_pred_proba)

    for i in range(n_bootstraps):
        indices = resample(np.arange(len(y_true_np)), replace=True, random_state=i)
        y_true_boot, y_pred_proba_boot = y_true_np[indices], y_pred_proba_np[indices]
        if len(np.unique(y_true_boot)) < 2: continue
        
        bootstrapped_pr_aucs.append(average_precision_score(y_true_boot, y_pred_proba_boot))
        y_pred_bin_boot = (y_pred_proba_boot >= threshold).astype(int)
        bootstrapped_f2_scores.append(fbeta_score(y_true_boot, y_pred_bin_boot, beta=2, zero_division=0))

    return bootstrapped_pr_aucs, bootstrapped_f2_scores

pr_auc_dist, f2_dist = calculate_bootstrap_ci(y, oof_preds, best_threshold)

In [ ]:
def bootstrap_business_metrics(df, probabilities, threshold, n_bootstraps=100, ci=95):
    df_stat = df[['channel_id', 'target']].copy()
    df_stat['pred'] = (probabilities >= threshold).astype(int)

    channel_summary = df_stat.groupby('channel_id').agg(is_broken=('target', 'max'), alarm_hours=('pred', 'sum')).reset_index()
    healthy_channels = channel_summary[channel_summary['is_broken'] == 0]
    broken_channels = channel_summary[channel_summary['is_broken'] == 1]

    fp_rates, avg_fp_hours, tp_rates, avg_tp_hours = [], [], [], []

    for i in range(n_bootstraps):
        boot_healthy = resample(healthy_channels, replace=True, random_state=i)
        boot_broken = resample(broken_channels, replace=True, random_state=i)

        n_healthy, n_broken = len(boot_healthy), len(boot_broken)
        fp_mask, tp_mask = boot_healthy['alarm_hours'] > 0, boot_broken['alarm_hours'] > 0
        count_fp, count_tp = fp_mask.sum(), tp_mask.sum()

        fp_rates.append((count_fp / n_healthy) * 100 if n_healthy > 0 else 0)
        avg_fp_hours.append(boot_healthy.loc[fp_mask, 'alarm_hours'].mean() if count_fp > 0 else 0)
        tp_rates.append((count_tp / n_broken) * 100 if n_broken > 0 else 0)
        avg_tp_hours.append(boot_broken.loc[tp_mask, 'alarm_hours'].mean() if count_tp > 0 else 0)

bootstrap_business_metrics(df, oof_preds, best_threshold)

In [ ]:
def plot_model_errors(sensor_id, df_plot, probabilities, threshold=0.5):
    mask = df_plot['channel_id'] == sensor_id
    sensor_df = df_plot[mask].copy()
    if len(sensor_df) == 0: return

    sensor_df['timestamp'] = pd.to_datetime(sensor_df['timestamp'])
    sensor_df['prob'] = probabilities[mask]
    sensor_df['prediction'] = (sensor_df['prob'] >= threshold).astype(int)

    tp_mask = (sensor_df['target'] == 1) & (sensor_df['prediction'] == 1)
    fn_mask = (sensor_df['target'] == 1) & (sensor_df['prediction'] == 0)
    fp_mask = (sensor_df['target'] == 0) & (sensor_df['prediction'] == 1)

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    bg_feature = 'mean_1h' if 'mean_1h' in sensor_df.columns else sensor_df.columns[2]
    t_all = sensor_df['timestamp']

    if fp_mask.sum() > 0:
        fig.add_trace(go.Scatter(x=t_all, y=fp_mask.astype(int) * 1.05, mode='lines', line=dict(width=0), fill='tozeroy', fillcolor='rgba(255, 165, 0, 0.25)', line_shape='hv', name="Зона ложной тревоги", hoverinfo="skip"), secondary_y=True)

    fig.add_trace(go.Scatter(x=t_all, y=sensor_df[bg_feature], name=bg_feature, line=dict(color='black', width=1)), secondary_y=False)
    fig.add_trace(go.Scatter(x=t_all, y=sensor_df['prob'], name="ML Prob", line=dict(color='blue', width=2, dash='dot')), secondary_y=True)
    fig.add_trace(go.Scatter(x=[t_all.min(), t_all.max()], y=[threshold, threshold], name=f"Threshold ({threshold:.2f})", line=dict(color='grey', dash='dash')), secondary_y=True)

    if 1 in sensor_df['target'].values:
        dt_start = sensor_df[sensor_df['target'] == 1]['timestamp'].min()
        fig.add_trace(go.Scatter(x=[dt_start, dt_start], y=[0, 1.05], mode="lines", line=dict(color="red", width=2, dash="dash"), name="Начало деградации", hoverinfo="skip"), secondary_y=True)
        status = "🔴 НЕИСПРАВНЫЙ"
    else:
        status = "🟡 ИСПРАВНЫЙ (Ложные тревоги!)" if fp_mask.sum() > 0 else "🟢 ИСПРАВНЫЙ"

    if tp_mask.sum() > 0: fig.add_trace(go.Scatter(x=sensor_df[tp_mask]['timestamp'], y=sensor_df[tp_mask][bg_feature], mode='markers', name="TP", marker=dict(color='green', size=6)), secondary_y=False)
    if fn_mask.sum() > 0: fig.add_trace(go.Scatter(x=sensor_df[fn_mask]['timestamp'], y=sensor_df[fn_mask][bg_feature], mode='markers', name="FN", marker=dict(color='red', size=8, symbol='x')), secondary_y=False)
    if fp_mask.sum() > 0: fig.add_trace(go.Scatter(x=sensor_df[fp_mask]['timestamp'], y=sensor_df[fp_mask][bg_feature], mode='markers', name="FP", marker=dict(color='orange', size=8, symbol='triangle-up')), secondary_y=False)

    fig.update_layout(title_text=f"{status} | ID: {sensor_id}", height=500, hovermode="x unified")
    fig.update_yaxes(range=[0, 1.05], secondary_y=True)
    fig.show()

broken_list = df[df['target'] == 1]['channel_id'].unique()
working_list = [c for c in df[df['target'] == 0]['channel_id'].unique() if c not in broken_list]

if len(broken_list) > 0: plot_model_errors(random.choice(broken_list), df, oof_preds, best_threshold)
if len(working_list) > 0:
    df_temp = df.copy()
    df_temp['pred_temp'] = (oof_preds >= best_threshold).astype(int)
    fp_sensors = df_temp[(df_temp['target'] == 0) & (df_temp['pred_temp'] == 1)]['channel_id'].unique()
    fp_working_sensors = [c for c in fp_sensors if c in working_list]
    sensor_to_plot = random.choice(fp_working_sensors) if len(fp_working_sensors) > 0 else random.choice(working_list)
    plot_model_errors(sensor_to_plot, df, oof_preds, best_threshold)

In [ ]:
def calculate_business_metrics(df, probabilities, threshold):
    df_stat = df[['channel_id', 'target']].copy()
    df_stat['pred'] = (probabilities >= threshold).astype(int)

    grouped = df_stat.groupby('channel_id').agg(is_broken=('target', 'max'), alarm_hours=('pred', 'sum')).reset_index()
    healthy_channels, broken_channels = grouped[grouped['is_broken'] == 0], grouped[grouped['is_broken'] == 1]

calculate_business_metrics(df, oof_preds, best_threshold)

In [ ]:
def benchmark_inference_speed(df, base_features, scaler, ae_model, lgbm_models, device):
    single_record = df[base_features].iloc[[0]].values
    batch_size = 1000
    batch_records = df[base_features].sample(n=batch_size, random_state=42, replace=True).values

    ae_model.eval()
    with torch.no_grad(): _ = ae_model(torch.FloatTensor(scaler.transform(single_record)).to(device))

    single_times = []
    for _ in range(100):
        start_time = time.perf_counter()
        tensor_x = torch.FloatTensor(scaler.transform(single_record)).to(device)
        with torch.no_grad(): mse = torch.mean((tensor_x - ae_model(tensor_x)) ** 2, dim=1).cpu().numpy()
        
        hybrid_x = np.column_stack((single_record, mse))
        ensemble_pred = np.zeros(len(hybrid_x))
        for model in lgbm_models: ensemble_pred += model.predict_proba(hybrid_x)[:, 1]
        
        single_times.append(time.perf_counter() - start_time)

    start_time = time.perf_counter()
    tensor_batch = torch.FloatTensor(scaler.transform(batch_records)).to(device)
    with torch.no_grad(): mse_batch = torch.mean((tensor_batch - ae_model(tensor_batch)) ** 2, dim=1).cpu().numpy()
    
    hybrid_batch = np.column_stack((batch_records, mse_batch))
    ensemble_preds_batch = np.zeros(len(hybrid_batch))
    for model in lgbm_models: ensemble_preds_batch += model.predict_proba(hybrid_batch)[:, 1]

benchmark_inference_speed(df, base_features, scaler, ae_model, final_models, device)

In [ ]:
def evaluate_on_unseen_data(df_new, base_features, scaler, ae_model, lgbm_models, threshold, device):
    X_new_scaled = scaler.transform(df_new[base_features].values)
    tensor_new = torch.FloatTensor(X_new_scaled).to(device)

    ae_model.eval()
    with torch.no_grad():
        mse_errors = torch.mean((tensor_new - ae_model(tensor_new)) ** 2, dim=1).cpu().numpy()

    df_new['reconstruction_error'] = mse_errors
    X_hybrid = df_new[base_features + ['reconstruction_error']]

    ensemble_preds = np.zeros(len(X_hybrid))
    for model in lgbm_models: ensemble_preds += model.predict_proba(X_hybrid)[:, 1]

    df_new['prob'] = ensemble_preds / len(lgbm_models)
    df_new['pred'] = (df_new['prob'] >= threshold).astype(int)
    
    return df_new

unseen_results_df = evaluate_on_unseen_data(unseen_df, base_features, scaler, ae_model, final_models, best_threshold, device)